# Modelo de Clasificación de Plagio — Iteración 2

Este notebook entrena el clasificador que decide si un **par de archivos de código** es plagio (1) o no plagio (0). El modelo no lee código directamente: recibe un **vector de 9 características** de similitud calculadas por el pipeline (`ast_tokenizer → winnowing → features`).

## Qué cambió respecto a la iteración 1 (F1 = 0.198, AUC = 0.534)

El diagnóstico de la iteración 1 reveló que el problema **no era el modelo, sino los datos**:

1. **Etiquetado imposible.** Las 20 "variantes" por problema del dataset de Kaggle son **algoritmos distintos** (recursivo vs. iterativo vs. fórmula de Binet...), no ofuscaciones unas de otras. Detectar esa equivalencia semántica está explícitamente *fuera del alcance* del proyecto (sección 9 del marco de referencia). El 75% de los pares positivos tenía similitud Winnowing = 0: no había señal que aprender.
2. **Solución (sección 5.3 del marco):** dataset **ad hoc con transformaciones documentadas**. Cada snippet original genera 4 variantes plagiadas mediante ofuscaciones reales (renombrado, código muerto, reescrituras equivalentes, reordenamiento — `pipeline/transforms.py`). Plagio = par del mismo grupo fuente. La mitad de los negativos son *difíciles*: mismo problema, algoritmo distinto.
3. **k = 15 en lugar de 23.** Los snippets tienen una mediana de ~73 tokens; un barrido empírico sobre k en {5,10,15,23,30} × w en {4,8,16} confirmó k=15, consistente con la hipótesis H1 del marco (k en {15, 23}).
4. **9 características en lugar de 5**, añadiendo métricas robustas en archivos cortos (Jaccard de k-gramas pequeños, coseno de histogramas de nodos AST, similitud de secuencia, razón de longitudes).
5. **Protocolo del marco (sección 8.2):** ajuste del umbral de decisión que maximiza F1 sobre validación, y comparación de algoritmos de ML (red neuronal vs. Random Forest, SVM y Regresión Logística — variable independiente de la sección 5.1).

**Metas del marco (sección 8.3):** Precisión ≥ 0.80 · Recall ≥ 0.75 · F1 ≥ 0.80 · AUC-ROC ≥ 0.85 · Línea base Dolos F1 = 0.865.

## 1. Importar librerías

TensorFlow para la red neuronal, scikit-learn para la validación cruzada, la normalización y los clasificadores de comparación.

In [1]:
import json
import os
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

sys.path.insert(0, ".")   # para importar pipeline/ desde el notebook
from pipeline.features import FEATURE_COLUMNS

## 2. Configuración

Semilla para reproducibilidad y parámetros del pipeline. `FEATURE_COLUMNS` (9 características) se importa directamente de `pipeline/features.py` para que el notebook y el pipeline nunca se desincronicen.

Los parámetros `K=15, W=4` fueron elegidos con un barrido sobre k en {5,10,15,23,30} × w en {4,8,16} (F1 con regresión logística, validación cruzada de 5 folds); k=15 está dentro de la hipótesis H1 del marco. El CSV `pairs_features.csv` debe generarse con esos mismos parámetros:

```bash
python -m pipeline.generate_dataset --output-dir data --variants 4
python -m pipeline.build_pairs --data-dir data --output pairs_features.csv --k 15 --w 4 --masking medium
```

In [2]:
RANDOM_SEED = 42
LABEL_COLUMN = "label"

# Parámetros del pipeline con los que se generó pairs_features.csv
K, W, MASKING = 15, 4, "medium"

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"{len(FEATURE_COLUMNS)} características: {FEATURE_COLUMNS}")

9 características: ['winnowing_similarity', 'shared_fragment_coverage', 'token_overlap_ratio', 'ast_depth_difference', 'fingerprint_jaccard', 'small_kgram_jaccard', 'node_type_cosine', 'token_sequence_ratio', 'length_ratio']


## 3. Cargar los datos

Si existe `pairs_features.csv` (generado por el pipeline) lo usamos. Si no, generamos datos sintéticos solo para probar que el modelo funciona.

In [3]:
def load_dataset(csv_path):
    df = pd.read_csv(csv_path)
    x = df[FEATURE_COLUMNS].values.astype("float32")
    y = df[LABEL_COLUMN].values.astype("float32")
    return x, y


def generate_synthetic_dataset(n_samples=2000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    n_pos = n_samples // 2
    n_neg = n_samples - n_pos
    pos = rng.normal(loc=0.8, scale=0.12, size=(n_pos, len(FEATURE_COLUMNS)))
    neg = rng.normal(loc=0.25, scale=0.12, size=(n_neg, len(FEATURE_COLUMNS)))
    x = np.clip(np.vstack([pos, neg]), 0.0, 1.0).astype("float32")
    y = np.concatenate([np.ones(n_pos), np.zeros(n_neg)]).astype("float32")
    idx = rng.permutation(n_samples)
    return x[idx], y[idx]


csv_path = "pairs_features.csv"
if os.path.exists(csv_path):
    print(f"Loading dataset from {csv_path}")
    x, y = load_dataset(csv_path)
else:
    print("Dataset not found, using synthetic data")
    x, y = generate_synthetic_dataset()

print(x.shape, y.shape)
print(f"Positivos: {int(y.sum())}  Negativos: {int((1 - y).sum())}")

Loading dataset from pairs_features.csv
(2000, 9) (2000,)
Positivos: 1000  Negativos: 1000


## 4. Construir el modelo

Es una red neuronal pequeña: dos capas ocultas y una salida con `sigmoid` que da una probabilidad entre 0 y 1. El entrenamiento usa *early stopping* sobre la pérdida de validación.

Medimos las métricas que pide el proyecto: precisión, recall y AUC-ROC.

In [4]:
def build_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    return model


def make_callbacks():
    """Early stopping: corta el entrenamiento cuando val_loss deja de
    mejorar y restaura los mejores pesos."""
    return [keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True,
    )]


def f1_from(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


build_model(len(FEATURE_COLUMNS)).summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 865 (3.38 KB)

 Trainable params: 865 (3.38 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Validación cruzada (5-fold)

Protocolo del marco (sección 8.2): K-fold con K=5. Dividimos los datos en 5 partes y entrenamos 5 veces, usando cada parte como validación una vez. Antes de entrenar, normalizamos las características (el scaler se ajusta solo con el fold de entrenamiento).

In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(x, y), start=1):
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x[train_idx])
    x_val = scaler.transform(x[val_idx])
    model = build_model(x.shape[1])
    model.fit(
        x_train, y[train_idx],
        validation_data=(x_val, y[val_idx]),
        epochs=100,
        batch_size=32,
        callbacks=make_callbacks(),
        verbose=0,
    )
    metrics = model.evaluate(x_val, y[val_idx], verbose=0, return_dict=True)
    metrics["f1"] = f1_from(metrics["precision"], metrics["recall"])
    scores.append(metrics)
    print(
        f"Fold {fold}: "
        f"acc={metrics['accuracy']:.3f} "
        f"precision={metrics['precision']:.3f} "
        f"recall={metrics['recall']:.3f} "
        f"f1={metrics['f1']:.3f} "
        f"auc={metrics['auc']:.3f}"
    )

Fold 1: acc=0.983 precision=0.975 recall=0.990 f1=0.983 auc=0.995


Fold 2: acc=0.990 precision=0.980 recall=1.000 f1=0.990 auc=1.000


Fold 3: acc=0.985 precision=0.975 recall=0.995 f1=0.985 auc=1.000


Fold 4: acc=0.985 precision=0.980 recall=0.990 f1=0.985 auc=0.993


Fold 5: acc=0.983 precision=0.975 recall=0.990 f1=0.983 auc=0.996


## 6. Resumen de la validación cruzada

Promediamos las métricas de los 5 folds para tener un número final por métrica.

In [6]:
for key in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = np.array([s[key] for s in scores])
    print(f"{key}: {values.mean():.3f} +/- {values.std():.3f}")

accuracy: 0.985 +/- 0.003
precision: 0.977 +/- 0.002
recall: 0.993 +/- 0.004
f1: 0.985 +/- 0.003
auc: 0.997 +/- 0.003


## 6b. Comparación de algoritmos de clasificación

El marco de referencia (sección 5.1) define el algoritmo de ML como variable independiente: Random Forest, SVM y Regresión Logística. Los comparamos con la misma validación cruzada de 5 folds para decidir si la red neuronal es la mejor opción.

In [7]:
classifiers = {
    "Regresión Logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED),
    "SVM (RBF)":           make_pipeline(StandardScaler(), SVC(random_state=RANDOM_SEED)),
}

scoring = ["precision", "recall", "f1", "roc_auc"]
comparison = []

for name, clf in classifiers.items():
    cv_res = cross_validate(clf, x, y, cv=skf, scoring=scoring)
    comparison.append({
        "modelo": name,
        "precision": cv_res["test_precision"].mean(),
        "recall": cv_res["test_recall"].mean(),
        "f1": cv_res["test_f1"].mean(),
        "auc": cv_res["test_roc_auc"].mean(),
    })

# Añadir la red neuronal (promedios de la sección 5)
comparison.append({
    "modelo": "Red Neuronal (TF)",
    "precision": np.mean([s["precision"] for s in scores]),
    "recall": np.mean([s["recall"] for s in scores]),
    "f1": np.mean([s["f1"] for s in scores]),
    "auc": np.mean([s["auc"] for s in scores]),
})

pd.DataFrame(comparison).set_index("modelo").round(3)

,precision,recall,f1,auc
modelo,,,,
Regresión Logística,0.977,0.988,0.983,0.997
Random Forest,0.981,0.998,0.990,0.997
SVM (RBF),0.975,0.992,0.984,0.994
Red Neuronal (TF),0.977,0.993,0.985,0.997


## 7. Modelo final, umbral de decisión y prueba

Entrenamos el modelo final con el 80% de los datos y lo evaluamos en el 20% restante, que nunca vio durante el entrenamiento.

Siguiendo el protocolo del marco (sección 8.2), el **umbral de decisión** no se fija en 0.5: se busca el que maximiza el F1 sobre el conjunto de validación, y ese umbral se aplica al conjunto de prueba. Las metas son Precisión ≥ 0.80, Recall ≥ 0.75, F1 ≥ 0.80 y AUC ≥ 0.85 (línea base Dolos: F1 = 0.865).

In [8]:
# 80% entrenamiento / 20% prueba (la prueba no se toca hasta el final)
x_train_full, x_test, y_train_full, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)
# Del entrenamiento, separar validación para early stopping y umbral
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.2,
    stratify=y_train_full, random_state=RANDOM_SEED,
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

model = build_model(x.shape[1])
model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=make_callbacks(),
    verbose=0,
)

# ── Umbral óptimo: maximiza F1 sobre validación (sección 8.2 del marco) ──
val_probs = model.predict(x_val, verbose=0).ravel()
thresholds = np.linspace(0.05, 0.95, 91)
f1_by_threshold = np.array([f1_score(y_val, val_probs >= t) for t in thresholds])
# Si varios umbrales empatan en F1 máximo, elegir el más cercano a 0.5
candidates = thresholds[f1_by_threshold >= f1_by_threshold.max() - 1e-9]
best_threshold = float(candidates[np.argmin(np.abs(candidates - 0.5))])
print(f"Umbral óptimo (validación): {best_threshold:.2f} "
      f"(F1 val = {f1_by_threshold.max():.3f})")

# ── Evaluación final en prueba con el umbral ajustado ──
test_probs = model.predict(x_test, verbose=0).ravel()
y_pred = (test_probs >= best_threshold).astype(int)

final_metrics = {
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "auc": roc_auc_score(y_test, test_probs),
}

print("\n── Resultados en prueba (20%, nunca vista) ──")
metas = {"precision": 0.80, "recall": 0.75, "f1": 0.80, "auc": 0.85}
for name, value in final_metrics.items():
    status = "CUMPLE" if value >= metas[name] else "NO CUMPLE"
    print(f"  {name:10s}: {value:.3f}   (meta >= {metas[name]:.2f}) {status}")
print(f"\nLínea base Dolos: F1 = 0.865 -> "
      f"{'superada' if final_metrics['f1'] >= 0.865 else 'NO superada'}")

Umbral óptimo (validación): 0.88 (F1 val = 0.984)



── Resultados en prueba (20%, nunca vista) ──
  precision : 0.985   (meta >= 0.80) CUMPLE
  recall    : 0.960   (meta >= 0.75) CUMPLE
  f1        : 0.972   (meta >= 0.80) CUMPLE
  auc       : 0.999   (meta >= 0.85) CUMPLE

Línea base Dolos: F1 = 0.865 -> superada


## 8. Guardar el modelo

Guardamos el modelo, el `StandardScaler` y el umbral de decisión: los tres son necesarios para predecir pares nuevos (ver `pipeline_demo.ipynb`).

In [9]:
import joblib

model.save("plagiarism_model.keras")
joblib.dump(scaler, "scaler.joblib")
with open("decision_threshold.json", "w") as f:
    json.dump({
        "threshold": best_threshold,
        "k": K, "w": W, "masking": MASKING,
        "feature_columns": FEATURE_COLUMNS,
        "test_metrics": {k_: round(v, 4) for k_, v in final_metrics.items()},
    }, f, indent=2)

print("Guardado: plagiarism_model.keras, scaler.joblib, decision_threshold.json")

Guardado: plagiarism_model.keras, scaler.joblib, decision_threshold.json
